# Ad Auction Mechanisms & Smart Bidding

Companion notebook for the [Ad Auction Mechanisms wiki page](https://ml-viz-ruby.vercel.app/wiki/ad-auction-and-bidding).

We implement Vickrey second-price auction, GSP multi-slot auction, and a simple budget pacing PI controller.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(3)

## 1 — Second-price (Vickrey) auction

In [ ]:
def vickrey_auction(bids, increment=0.01):
    order = np.argsort(-bids)
    winner = order[0]
    payment = bids[order[1]] + increment
    return winner, payment

# Show truthfulness: any bid < true value is dominated
true_value = 10.0
highest_opponent = 7.0

outcomes = []
for bid in [5.0, 7.0, 9.0, 10.0, 12.0, 15.0]:
    win = bid > highest_opponent
    pay = highest_opponent + 0.01 if win else 0.0
    profit = (true_value - pay) if win else 0.0
    outcomes.append((bid, win, pay, profit))
    
print(f"True value = {true_value}, Highest opponent = {highest_opponent}")
print(f"{'Bid':>6s} | {'Win?':>5s} | {'Payment':>8s} | {'Profit':>8s}")
for bid, win, pay, profit in outcomes:
    print(f"{bid:>6.1f} | {str(win):>5s} | {pay:>8.2f} | {profit:>8.2f}")
print("\nBidding true value (10.0) maximizes profit → dominant strategy")

## 2 — GSP multi-slot auction

In [ ]:
def gsp_auction(bid_quality_scores, slots=3, increment=0.01):
    """
    Generalized Second Price auction for multiple slots.
    bid_quality_scores: array of bid * quality for each advertiser
    Returns (winner_ids, payments)
    """
    order = np.argsort(-bid_quality_scores)
    winners = order[:slots]
    payments = []
    for k in range(slots):
        if k + 1 < len(bid_quality_scores):
            payments.append(bid_quality_scores[order[k+1]] + increment)
        else:
            payments.append(increment)
    return winners, np.array(payments)

advertisers = ['Nike', 'Adidas', 'Puma', 'Reebok', 'New Balance']
bids = np.array([5.0, 3.5, 4.2, 2.8, 1.5])
quality = np.array([0.8, 0.9, 0.6, 0.7, 0.95])   # expected CTR / quality score
bqs = bids * quality

winners, payments = gsp_auction(bqs, slots=3)
slot_ctrs = np.array([0.10, 0.06, 0.03])  # slot click-through rates

print(f"{'Slot':>5s} | {'Advertiser':>12s} | {'BxQ':>6s} | {'Payment':>8s} | {'Net ROI @$5 value':>18s}")
for k, (w, pay) in enumerate(zip(winners, payments)):
    roi = (5.0 - pay/quality[w]) * slot_ctrs[k] * 1000
    print(f"{k+1:>5d} | {advertisers[w]:>12s} | {bqs[w]:>6.2f} | ${pay:>7.2f} | ${roi:>16.2f}")

## 3 — Budget pacing PI controller

In [ ]:
class PacingController:
    def __init__(self, daily_budget, Kp=0.1, Ki=0.01):
        self.daily_budget = daily_budget
        self.Kp, self.Ki = Kp, Ki
        self.rho = 1.0     # bid multiplier (1 = full bid)
        self.integral = 0.0

    def update(self, actual_spend, target_spend):
        """Update multiplier based on spend error."""
        error = target_spend - actual_spend   # positive = underspending
        self.integral += error
        adjustment = self.Kp * error + self.Ki * self.integral
        self.rho = np.clip(self.rho + adjustment, 0.1, 1.0)
        return self.rho

# Simulate a 24-hour pacing cycle with 48 half-hour intervals
budget = 1000.0
controller = PacingController(daily_budget=budget)
target_rate = budget / 48  # equal pacing

rhos, spends = [1.0], [0.0]
total_spend = 0.0
for i in range(48):
    # Random spend: noisy around target (morning spike, afternoon slump)
    noise = rng.normal(0, target_rate * 0.3)
    time_factor = 1.3 if 12 <= i <= 24 else 0.7  # morning/afternoon variation
    raw_spend = max(0, target_rate * time_factor * controller.rho + noise)
    total_spend += raw_spend
    target_cumulative = target_rate * (i+1)
    rho = controller.update(total_spend, target_cumulative)
    rhos.append(rho)
    spends.append(total_spend)

print(f"Total spend: ${total_spend:.2f} / ${budget:.2f} ({100*total_spend/budget:.1f}%)")
print(f"Final bid multiplier: {rhos[-1]:.3f}")

## ✏️ Your turn

In [ ]:
def optimal_bid(true_value, target_cpa, p_conversion):
    """
    Compute the optimal bid for a Target CPA campaign.
    The bid should equal: target_CPA × P(conversion | context)
    
    true_value: advertiser's value per conversion (e.g., $50)
    target_cpa: maximum cost-per-acquisition the advertiser will pay
    p_conversion: predicted conversion probability for this impression
    Returns: optimal bid
    """
    # TODO(you): return target_cpa * p_conversion
    return ...

cases = [(50, 30, 0.05), (100, 40, 0.02), (200, 60, 0.10)]
print(f"{'Value':>8s} | {'Target CPA':>10s} | {'P(conv)':>8s} | {'Optimal bid':>12s} | {'Worthwhile?':>12s}")
for val, cpa, p_conv in cases:
    bid = optimal_bid(val, cpa, p_conv)
    worthwhile = bid > 0 and (val * p_conv) > bid
    print(f"  ${val:>5.0f}  |    ${cpa:>6.0f}   |  {p_conv:.3f}  |   ${bid:>9.3f}  | {worthwhile}")

<details><summary>Solution</summary>

```python
def optimal_bid(true_value, target_cpa, p_conversion):
    return target_cpa * p_conversion
```

The bid = target_CPA × P(conversion) is the expected CPA at this bid, which should equal the target. If P(conversion) = 0.05 and target CPA = $30, bid = $1.50 per click — you'd expect 1 conversion per 20 clicks, costing $30.
</details>